In [ ]:
import numpy as np
from gould_2026.datasets import Zong22Dataset
from sim_stim import make_srs
from matplotlib.legend_handler import HandlerPatch
import matplotlib.patches as mpatches
import pathlib
import matplotlib.pyplot as plt
from gould_2026.stim_designer import StimDesigner, OptimizationMethod
from tqdm.autonotebook import tqdm
from IPython.display import display, clear_output



In [ ]:
output = None

In [ ]:
rng = np.random.default_rng(0)
d = Zong22Dataset()
data = d.neural_data

srs = make_srs(data, rng, comparison_preset='visualization', n_runs=1, show_tqdm=True)


i= 40
sr = srs['learning from stim'][0]

fig, axs = plt.subplots(ncols=2, figsize=(10,4), sharex=False, sharey=False, layout='constrained')

latents = sr.log['latents'].slice_by_time(slice(30,None))
axs[0].plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
stim_s = sr.log['stim_intended_samples'].t - latents.dt

l = 1
r = 5.1
ax_n = 0
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = axs[ax_n].plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
axs[ax_n].plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')

for arrow_index in [17, 50]:
    axs[0].annotate('',
                    xytext=(latents[arrow_index, 0], latents[arrow_index, 1]),
                    xy=(latents[arrow_index+1, 0], latents[arrow_index+1, 1]),
                    arrowprops=dict(arrowstyle="simple", color='C0'),
                    size=11
                    )


u = sr.stim_designer.log[i]['u']
idx = np.argsort(np.abs(u))[::-1]
print(np.linalg.norm(u,ord=0))

high_d = sr.log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
axs[1].plot(high_d.t, high_d[:,idx[:int(np.linalg.norm(u,ord=0))]]);
for stim_t in stim_s:
    axs[1].axvline(stim_t, color='r')


In [ ]:
fig, ax = plt.subplots()


latents = sr.log['latents'].slice_by_time(slice(30,None))
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

i = 40
l = 1
r = 5.1
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')


In [ ]:
colors = {
    'blue':f'#00274C',
    'maize':'#1e7608ff',
    'lqs':'#cccccc',
    'red':'#9A3324',
    'orange':'#D86018',
    'su_blue':'#174992',
}


In [ ]:
fig, axs = plt.subplots(figsize=(5,5), layout='constrained',squeeze=False)


stim_designer = StimDesigner(should_log=True, rng_seed=1)

theta=.1
i=40
l=1
r=0
opt_method=OptimizationMethod.JAXOPT

latents = sr.log['latents'].slice_by_time(slice(30,None))
ax = axs[0,0]
ax.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')

center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = ax.plot(latents[:-1, 0], latents[:-1, 1], color='C0')
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
ax.plot(latents_s[:, 0], latents_s[:, 1], '.', color='g')



u = sr.stim_designer.log[i]['u']
v = sr.stim_designer.log[i]['v']

for theta in np.linspace(0, 2*np.pi, 20) - 0.11:
    v = 0 * v
    v[0,0] = np.cos(theta)
    v[1,0] = np.sin(theta)

    equivalent_projection_matrix = sr.stim_designer.log[i]['equiv_proj_mat']
    u_to_s_function=lambda u: equivalent_projection_matrix.T @ u


    stim_designer.optimization_method = opt_method
    new_u = stim_designer.design_stim(v=v, u_dimension=u.size, u_to_s_function=u_to_s_function, equivalent_projection_matrix=equivalent_projection_matrix)

    # v_arrow = ax.annotate('',
    #                       xytext=(latents_s[0,0], latents_s[0,1]),
    #                       xy=(latents_s[0,0]+v[0,0], latents_s[0,1]+v[1,0]),
    #                       arrowprops=dict(color=colors['orange'], width=1.5),
    #                       size=20
    #                       )

    s = u_to_s_function(new_u)*4
    s_marker = axs[0,0].annotate('',
                               xytext=(latents_s[0,0], latents_s[0,1]),
                               xy=(latents_s[0,0]+s[0], latents_s[0,1]+s[1]),
                               arrowprops=dict(color=colors['maize'], width=1.5),
                               size=20,
                               ).arrow_patch





    def make_legend_arrow(legend, orig_handle,
                          xdescent, ydescent,
                          width, height, fontsize):
        p = mpatches.FancyArrow(width, 0.5*height, -width, 0, length_includes_head=True, head_width=0.7*height, head_length=.23*width)
        return p


axs[0,0].axis('equal');
axs[0,0].axis('off');
axs[0,0].set_xlim(np.array([-2,2]) + 1.26)
axs[0,0].set_ylim(np.array([-2,2]) + 0.73)



if output is not None:
    fig.savefig(output, dpi=400)
